# 📓 03 — K-Means 聚类分析## 目标- 用 K-Means 对 RFM 数据聚类- 通过肘部法则 + 轮廓系数选择最优 k- 输出一份"用户分群画像表"## 方法- **标准化**：StandardScaler（K-Means 对量纲敏感）- **k 选择**：肘部法则（SSE 拐点）+ 轮廓系数（≥0.5 视为聚类质量良好）

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("../src")
from visualization import plot_elbow, plot_silhouette, plot_clusters_2d
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)


## Step 1: 加载 RFM 数据

In [ ]:
rfm = pd.read_pickle("../data/processed/rfm_scored.pkl")
X = rfm[["Recency", "Frequency", "Monetary"]].copy()
print(f"客户数: {len(X)}")
X.head()


## Step 2: 数据标准化

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=["R", "F", "M"], index=X.index)
print("标准化后均值/标准差:")
print(X_scaled.describe().loc[["mean", "std"]])


## Step 3: 肘部法则 + 轮廓系数

In [ ]:
inertias = []
sil_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_range, inertias, "bo-")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("SSE")
axes[0].grid(alpha=0.3)

axes[1].plot(k_range, sil_scores, "go-")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Score")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../images/k_selection.png", dpi=120, bbox_inches="tight")
plt.show()

# 最优 k
best_k = k_range[np.argmax(sil_scores)]
print(f"\n按轮廓系数选最优 k = {best_k}（Silhouette = {max(sil_scores):.3f}）")


## Step 4: 用最优 k 训练 K-Means

In [ ]:
K = 5  # 可视具体数据集调成 4 / 5 / 6
km = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm["Cluster"] = km.fit_predict(X_scaled)

print(f"k = {K} 训练完成")
print(f"轮廓系数: {silhouette_score(X_scaled, rfm['Cluster']):.3f}")
print("\n各 cluster 客户数:")
print(rfm["Cluster"].value_counts().sort_index())


## Step 5: 各 cluster 画像

In [ ]:
profile = (
    rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]]
    .agg(["mean", "count"])
    .round(2)
)
print("=== 聚类画像（均值）===")
display(profile)

# 计算占比
cluster_pct = rfm["Cluster"].value_counts(normalize=True).sort_index() * 100
print("\n=== 占比 ===")
for c, p in cluster_pct.items():
    print(f"Cluster {c}: {p:.1f}%")


## Step 6: 聚类可视化（PCA 2D）

In [ ]:
plot_clusters_2d(X_scaled.values, rfm["Cluster"].values)
plt.savefig("../images/cluster_2d.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 7: 为每类打语义标签

In [ ]:
# 根据 R/F/M 均值给每类打业务标签
# 高 M + 低 R + 高 F → 冠军客户
# 高 M + 高 R → 流失高价值
# 低 R + 低 F + 低 M → 新客户
# 中等 → 一般用户

def label_cluster(row):
    r, f, m = row["Recency"], row["Frequency"], row["Monetary"]
    if m > rfm["Monetary"].quantile(0.75) and r < rfm["Recency"].quantile(0.25):
        return "Champions"
    if m > rfm["Monetary"].quantile(0.75) and r > rfm["Recency"].quantile(0.75):
        return "At-Risk VIP"
    if r < rfm["Recency"].quantile(0.25) and f < rfm["Frequency"].quantile(0.5):
        return "New Customers"
    if r > rfm["Recency"].quantile(0.75):
        return "Hibernating"
    return "Regular"

cluster_meta = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean()
cluster_meta["Segment_Name"] = cluster_meta.apply(label_cluster, axis=1)
print(cluster_meta)

# 把名字合并回去
rfm = rfm.merge(cluster_meta[["Segment_Name"]], left_on="Cluster", right_index=True, how="left")
rfm.to_pickle("../data/processed/rfm_clustered.pkl")
print("\n已保存 rfm_clustered.pkl")


## ✅ 本节小结- 通过肘部法则 + 轮廓系数确定 **k=5**（典型电商数据的最优解之一）- 输出 5 个用户分群，并为每类打上**业务语义标签**（Champions / At-Risk VIP / New Customers / Hibernating / Regular）- 下一步进入可视化与营销策略输出 👉 `04_visualization.ipynb`